# **Day 59: The collections Module**

The `collections` module gives us **special containers** that make common jobs much easier.

| Name | What it does |
|---|---|
| `Counter` | Counts things |
| `defaultdict` | A dictionary that gives a default value for missing keys |
| `namedtuple` | A tuple with names for each position |
| `deque` | A fast list for adding/removing at **both** ends |
| `OrderedDict` | A dictionary with extra ordering tools |
| `ChainMap` | Searches many dictionaries as one |

## **Counter**

In [1]:
from collections import Counter

votes = "tmc bjp tmc cpim tmc congress bjp tmc cpim bjp tmc".split()
c = Counter(votes)
print(c)
print(c.most_common(2))
print(c["tmc"], c["aap"])           # A missing key gives 0, not an error

Counter({'tmc': 5, 'bjp': 3, 'cpim': 2, 'congress': 1})
[('tmc', 5), ('bjp', 3)]
5 0


In [2]:
morning = Counter(rosogolla=40, sandesh=25)
evening = Counter(rosogolla=55, sandesh=10, langcha=12)
print(morning + evening)            # Counters can be added

Counter({'rosogolla': 95, 'sandesh': 35, 'langcha': 12})


## **Example from Remote Sensing: Land Cover Statistics and Change**
`Counter` is perfect for counting the pixels of each class in a classified image. Let us compare the land cover of **New Town, Kolkata** in two years to see how it changed. (The pixel data is made up.)

In [3]:
landcover_2010 = (["Agriculture"] * 420 + ["Water"] * 160 + ["Built-up"] * 250
                  + ["Vegetation"] * 130 + ["Bare land"] * 40)
landcover_2024 = (["Agriculture"] * 150 + ["Water"] * 110 + ["Built-up"] * 560
                  + ["Vegetation"] * 120 + ["Bare land"] * 60)

c2010 = Counter(landcover_2010)
c2024 = Counter(landcover_2024)

pixel_ha = 0.09                    # One 30 m Landsat pixel = 900 sq m = 0.09 hectare
print(f"{'Class':<12}{'2010 (ha)':>10}{'2024 (ha)':>10}{'Change':>9}")
for cls in c2010 | c2024:
    a, b = c2010[cls] * pixel_ha, c2024[cls] * pixel_ha
    print(f"{cls:<12}{a:>10.1f}{b:>10.1f}{b - a:>+9.1f}")

Class        2010 (ha) 2024 (ha)   Change
Agriculture       37.8      13.5    -24.3
Water             14.4       9.9     -4.5
Built-up          22.5      50.4    +27.9
Vegetation        11.7      10.8     -0.9
Bare land          3.6       5.4     +1.8


Let us understand the code above:
- `Counter(list)` counts how many times each class appears.
- `c2010 | c2024` gives all class names found in either year.
- Multiplying the pixel count by 0.09 changes pixels into **hectares**.
- The result shows **urban growth**: built-up area grew a lot, while agriculture and water decreased.

## **defaultdict**
A normal dictionary gives an error for a missing key. A `defaultdict` **creates** a default value automatically.

In [4]:
from collections import defaultdict

places = [("north", "Siliguri"), ("south", "Kolkata"), ("north", "Darjeeling"),
          ("south", "Digha"), ("west", "Purulia"), ("north", "Kalimpong")]

by_region = defaultdict(list)       # Missing keys start as an empty list
for region, p in places:
    by_region[region].append(p)
print(dict(by_region))

{'north': ['Siliguri', 'Darjeeling', 'Kalimpong'], 'south': ['Kolkata', 'Digha'], 'west': ['Purulia']}


## **namedtuple**
A tuple where each position also has a **name**. It is light and cannot be changed.

In [5]:
from collections import namedtuple

Station = namedtuple("Station", ["code", "name", "lat", "lon"])
s = Station("KGP", "Kharagpur", 22.3395, 87.3250)

print(s)
print(s.name, s.lat)       # Use by name
print(s[0])                # Or by position
print(s._asdict())

Station(code='KGP', name='Kharagpur', lat=22.3395, lon=87.325)
Kharagpur 22.3395
KGP
{'code': 'KGP', 'name': 'Kharagpur', 'lat': 22.3395, 'lon': 87.325}


## **deque (Double-Ended Queue)**
A `deque` can add and remove items **quickly from both ends**. Removing from the start of a normal list (`pop(0)`) is slow for big lists.

In [6]:
from collections import deque

q = deque(["Ram", "Shyam", "Jodu"])
q.append("Modhu")          # Join at the back
q.appendleft("VIP")        # Join at the front
print(q)
print(q.popleft(), q.pop())
print(q)

deque(['VIP', 'Ram', 'Shyam', 'Jodu', 'Modhu'])
VIP Modhu
deque(['Ram', 'Shyam', 'Jodu'])


In [7]:
# maxlen keeps only the last N items: a moving window of GPS speeds
recent_speeds = deque(maxlen=3)
for speed in [4.1, 18.5, 2.3, 22.0, 0.8, 3.9]:
    recent_speeds.append(speed)
    print(list(recent_speeds), "-> average", round(sum(recent_speeds) / len(recent_speeds), 1))

[4.1] -> average 4.1
[4.1, 18.5] -> average 11.3
[4.1, 18.5, 2.3] -> average 8.3
[18.5, 2.3, 22.0] -> average 14.3
[2.3, 22.0, 0.8] -> average 8.4
[22.0, 0.8, 3.9] -> average 8.9


## **ChainMap**
Looks for a key in many dictionaries, one after another. Useful for **settings**: user settings first, then default settings.

In [8]:
from collections import ChainMap

defaults = {"theme": "light", "lang": "en", "crs": "EPSG:4326"}
user = {"lang": "bn"}
settings = ChainMap(user, defaults)
print(settings["lang"], settings["theme"], settings["crs"])

bn light EPSG:4326


In [9]:
# Quick question
# Guess the output of this code cell
# from collections import Counter
# print(Counter("banana").most_common(1))